In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

In [2]:
# modules/trainer_xgb.py

import pandas as pd
import joblib
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import RocCurveDisplay
from sklearn.metrics import precision_recall_curve, average_precision_score

def train_xgboost(csv_path="data/asos_seoul_daily_enriched.csv", model_output="models/xgb_model_daily.pkl", scaler_output="models/xgb_scaler_daily.pkl"):
    df = pd.read_csv(csv_path) #xgboost용 데이터 읽기(trainer.py참조)

    features = [ # 16개 입력 변수 
        'avgTa', 'minTa', 'maxTa', 'sumRn', 'avgWs', 'avgRhm', 'avgTs', 'avgTd', 'avgPs',
        'month', 'day', 'weekday', 'is_weekend', 'is_rainy', 'rain_hours', 'max_hourly_rn'
    ]
    target = 'flood_risk'

    X = df[features] #독립변수:features
    y = df[target] #타겟변수:침수 위험도(0, 1)

    # 정규화
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X) #fit_transform()으로 스케일링 파라미터 학습과 변환을 한 번에 수행

    # 데이터 분할
    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, stratify=y, test_size=0.2, random_state=42) #훈련용:테스트용=80:20

    # 모델 학습
    model = xgb.XGBClassifier(
        n_estimators=100, #100개 트리 생성
        max_depth=4, #각 트리의 최대 갚이4(과적합 방지)
        learning_rate=0.1, #학습률0.1(보수적)
        use_label_encoder=False, #경고메세지 방지
        eval_metric='logloss', #이진 분류용 손실 함수
        random_state=42 #시드값 고정
    )
    model.fit(X_train, y_train) #모델 학습

    # 평가
    y_pred = model.predict(X_test) #0또는 1예측
    y_proba = model.predict_proba(X_test)[:, 1] #침수 위험도 확률(0~1사이)

    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred)) #Confusion Matrix: 실제 vs 예측의 교차표
    print("\nClassification Report:\n", classification_report(y_test, y_pred)) #Classification Report: 정밀도, 재현율, F1-score 등
    print("\nROC AUC Score:", roc_auc_score(y_test, y_proba)) #ROC AUC Score: 분류 성능 종합 지표

    # 저장
    joblib.dump(model, model_output)
    joblib.dump(scaler, scaler_output)
    print(f"모델 저장 완료: {model_output}")
    print(f"스케일러 저장 완료: {scaler_output}")

    # 시각화 저장

    #Confusion Matrix 히트맵 : 혼동 행렬을 히트맵으로 시각화
    plt.figure(figsize=(6, 4))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix (XGBoost)")
    plt.tight_layout()
    plt.savefig("outputs/xgb_confusion_matrix_daily.png")
    plt.show()

    #ROC Curve 시각화 : 모델의 분류 성능을 시각화
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("ROC Curve (XGBoost)")
    plt.savefig("outputs/xgb_roc_curve_daily.png")
    plt.show()

    # Precision-Recall 시각화 : 정밀도-재현율 곡선
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)

    plt.figure(figsize=(8, 6))
    plt.plot(recall, precision, label=f'AP = {avg_precision:.3f}')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curve (XGBoost)')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig("outputs/xgb_precision_recall_daily.png")
    plt.show()

    # Feature Importance 시각화 : 각 특성의 중요도를 gain 기준으로 시각화. 홍수 예측에 어떤 특성이 가장 중요한지 확인 가능
    plt.figure(figsize=(10, 6))
    xgb.plot_importance(model, importance_type='gain', max_num_features=15, height=0.5)
    plt.title('XGBoost Feature Importance (Top 15 by Gain)')
    plt.tight_layout()
    plt.savefig("outputs/xgb_feature_importance_daily.png")
    plt.show()
